In [1]:
import pandas as pd
import sqlite3

In [23]:
from google.colab import files

uploaded = files.upload()

Saving orders.csv to orders (1).csv
Saving products.csv to products (1).csv
Saving customers.csv to customers (1).csv


In [24]:
customers_df = pd.read_csv("customers.csv")
products_df = pd.read_csv("products.csv")
orders_df = pd.read_csv("orders.csv")

In [25]:
print(customers_df.head())
print(products_df.head())
print(orders_df.head())

   customer_id customer_name       city
0            1         Rahul      Delhi
1            2         Priya       Pune
2            3          Amit   Banglore
3            4          Neha     Mumbai
4            5        Anjali  Hyderabad
   product_id product_name     category
0         101       Laptop  Electronics
1         102    Headphone  Electronics
2         103        Shoes      Fashion
3         104        Watch  Accessories
4         105        Table    Furniture
   order_id  customer_id  product_id  order_date  amount
0         1            1         101  2025-01-05  50,000
1         2            2         102  2025-05-05  20,000
2         3            2         101  2025-11-06  50,000
3         4            3         104  2025-02-08    6500
4         5            4         103  2025-01-05  10,000


In [35]:
conn = sqlite3.connect("retail_sales.db")

cursor = conn.cursor()

In [36]:
orders_df["amount"] = (
    orders_df["amount"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .astype(float)
)

In [37]:
orders_df.to_sql(
    "orders",
    conn,
    if_exists="replace",
    index=False
)

7

In [19]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT,
    city TEXT
)
""")

In [20]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT
)
""")

In [21]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    product_id INTEGER,
    order_date DATE,
    amount REAL,
    FOREIGN KEY(customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY(product_id) REFERENCES products(product_id)
)
""")

In [30]:
pd.read_sql(
    "SELECT * FROM customers",
    conn
)

,customer_id,customer_name,city
0,1,Rahul,Delhi
1,2,Priya,Pune
2,3,Amit,Banglore
3,4,Neha,Mumbai
4,5,Anjali,Hyderabad


In [31]:
pd.read_sql(
    "SELECT * FROM products",
    conn
)

,product_id,product_name,category
0,101,Laptop,Electronics
1,102,Headphone,Electronics
2,103,Shoes,Fashion
3,104,Watch,Accessories
4,105,Table,Furniture


In [13]:
pd.read_sql(
    "SELECT * FROM products",
    conn
)

,product_id,product_name,category
0,101,Laptop,Electronics
1,102,Headphone,Electronics
2,103,Shoes,Fashion
3,104,Watch,Accessories
4,105,Table,Furniture


Join


In [32]:
query = """
SELECT
    c.customer_name,
    p.product_name,
    o.amount

FROM orders o

JOIN customers c
ON o.customer_id = c.customer_id

JOIN products p
ON o.product_id = p.product_id
"""

pd.read_sql(query, conn)

,customer_name,product_name,amount
0,Rahul,Laptop,50000.0
1,Priya,Headphone,20000.0
2,Priya,Laptop,50000.0
3,Amit,Watch,6500.0
4,Neha,Shoes,10000.0
5,Amit,Table,30000.0
6,Anjali,Headphone,20000.0


Window Function

In [33]:
query = """
WITH customer_spending AS
(
    SELECT
        c.customer_name,
        SUM(o.amount) AS total_spent

    FROM customers c
    JOIN orders o

    ON c.customer_id = o.customer_id

    GROUP BY c.customer_name
)

SELECT
    customer_name,
    total_spent,

    RANK() OVER
    (
        ORDER BY total_spent DESC
    ) AS spending_rank

FROM customer_spending
"""

pd.read_sql(query, conn)

,customer_name,total_spent,spending_rank
0,Priya,70000.0,1
1,Rahul,50000.0,2
2,Amit,36500.0,3
3,Anjali,20000.0,4
4,Neha,10000.0,5
